In [103]:
# drop rows where every financial field is null
financial_cols = ["revenue", "net_income", "total_assets", "total_equity", "total_debt", "eps"]
fundamentals_clean = fundamentals_clean.dropna(subset=financial_cols, how="all")

print(f"Rows after dropping empties: {len(fundamentals_clean)}")
print(f"Unique tickers: {fundamentals_clean['ticker'].nunique()}")
print()

# save
output_path = "/Users/jtee/Documents/us-equity-screener/data/fundamentals_raw.csv"
fundamentals_clean.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Rows after dropping empties: 406
Unique tickers: 101

Saved to /Users/jtee/Documents/us-equity-screener/data/fundamentals_raw.csv


In [104]:
# overall null counts per column
print("Nulls per column:")
print(fundamentals_clean.isnull().sum())
print()

# rows where every financial field is null (useless rows)
financial_cols = ["revenue", "net_income", "total_assets", "total_equity", "total_debt", "eps"]
all_null = fundamentals_clean[financial_cols].isnull().all(axis=1)
print(f"Rows with all financial fields null: {all_null.sum()}")
print()

# how many years of data does each ticker have (after dropping empty rows)
ticker_coverage = fundamentals_clean[~all_null].groupby("ticker").size().sort_values()
print("Tickers with fewest years of data:")
print(ticker_coverage.head(10))
print()
print(f"Median years per ticker: {ticker_coverage.median()}")

Nulls per column:
revenue         2
net_income      6
total_assets    2
total_equity    2
total_debt      5
eps             2
ticker          0
fiscal_year     0
dtype: int64

Rows with all financial fields null: 0

Tickers with fewest years of data:
ticker
AAPL    4
NOW     4
NKE     4
NFLX    4
NEE     4
MU      4
MSFT    4
MS      4
MRK     4
MO      4
dtype: int64

Median years per ticker: 4.0


In [105]:
# map of Yahoo's column names to the clean names we want
column_map = {
    "Total Revenue":        "revenue",
    "Net Income":           "net_income",
    "Total Assets":         "total_assets",
    "Stockholders Equity":  "total_equity",
    "Total Debt":           "total_debt",
    "Diluted EPS":          "eps",
}

# keep only columns that actually exist in the raw data
available = [col for col in column_map if col in fundamentals_raw.columns]
missing = [col for col in column_map if col not in fundamentals_raw.columns]

print(f"Available columns: {available}")
print(f"Missing columns: {missing}")

# slice down to wanted columns + ticker + fiscal_year, then rename
fundamentals_clean = fundamentals_raw[available + ["ticker", "fiscal_year"]].copy()
fundamentals_clean = fundamentals_clean.rename(columns=column_map)

print(fundamentals_clean.head())
print(f"\nShape: {fundamentals_clean.shape}")

Available columns: ['Total Revenue', 'Net Income', 'Total Assets', 'Stockholders Equity', 'Total Debt', 'Diluted EPS']
Missing columns: []
        revenue    net_income  total_assets  total_equity    total_debt  \
0  2.159380e+11  1.200670e+11  2.068030e+11  1.572930e+11  1.104000e+10   
1  1.304970e+11  7.288000e+10  1.116010e+11  7.932700e+10  9.982000e+09   
2  6.092200e+10  2.976000e+10  6.572800e+10  4.297800e+10  1.105600e+10   
3  2.697400e+10  4.368000e+09  4.118200e+10  2.210100e+10  1.203100e+10   
4           NaN           NaN           NaN           NaN           NaN   

     eps ticker  fiscal_year  
0  4.900   NVDA         2026  
1  2.940   NVDA         2025  
2  1.190   NVDA         2024  
3  0.174   NVDA         2023  
4    NaN   NVDA         2022  

Shape: (499, 8)


In [106]:
# stitch all 101 DataFrames into one
fundamentals_raw = pd.concat(fundamentals_list, ignore_index=True)

print(f"Raw rows: {len(fundamentals_raw)}")
print(f"Raw columns: {len(fundamentals_raw.columns)}")

Raw rows: 499
Raw columns: 217


In [107]:
fundamentals_list = []
failed = []

for ticker in all_tickers:
    try:
        t = yf.Ticker(ticker)
        income = t.financials.T
        balance = t.balance_sheet.T

        # combine income statement and balance sheet side by side
        df = pd.concat([income, balance], axis=1)

        # tag with the ticker and year
        df["ticker"] = ticker
        df["fiscal_year"] = df.index.year

        fundamentals_list.append(df)
        time.sleep(0.5)

    except Exception as e:
        failed.append((ticker, str(e)))

print(f"Successful: {len(fundamentals_list)} tickers")
print(f"Failed: {len(failed)} tickers")
if failed:
    print("Failed tickers:")
    for ticker, error in failed[:10]:
        print(f"  {ticker}: {error[:80]}")

Successful: 102 tickers
Failed: 0 tickers


In [108]:
test_ticker = yf.Ticker("AAPL")

income = test_ticker.financials.T
balance = test_ticker.balance_sheet.T

print("Income statement columns:")
print(income.columns.tolist())
print()
print("Balance sheet columns:")
print(balance.columns.tolist())
print()
print("Income statement shape:", income.shape)
print("Balance sheet shape:", balance.shape)

Income statement columns:
['Tax Effect Of Unusual Items', 'Tax Rate For Calcs', 'Normalized EBITDA', 'Net Income From Continuing Operation Net Minority Interest', 'Reconciled Depreciation', 'Reconciled Cost Of Revenue', 'EBITDA', 'EBIT', 'Net Interest Income', 'Interest Expense', 'Interest Income', 'Normalized Income', 'Net Income From Continuing And Discontinued Operation', 'Total Expenses', 'Total Operating Income As Reported', 'Diluted Average Shares', 'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Diluted NI Availto Com Stockholders', 'Net Income Common Stockholders', 'Net Income', 'Net Income Including Noncontrolling Interests', 'Net Income Continuous Operations', 'Tax Provision', 'Pretax Income', 'Other Income Expense', 'Other Non Operating Income Expenses', 'Net Non Operating Interest Income Expense', 'Interest Expense Non Operating', 'Interest Income Non Operating', 'Operating Income', 'Operating Expense', 'Research And Development', 'Selling General And Administration', 

In [120]:
output_path = "/Users/jtee/Documents/us-equity-screener/data/prices_raw.csv"
prices_long.to_csv(output_path, index=False)
print(f"Saved {len(prices_long)} rows to {output_path}")

Saved 7286 rows to /Users/jtee/Documents/us-equity-screener/data/prices_raw.csv


In [119]:
prices_long = (
    prices_full["Close"]
    .reset_index()
    .melt(id_vars="Date", var_name="ticker", value_name="close")
)

# drop rows with missing prices (tickers that don't have data for some months)
prices_long = prices_long.dropna(subset=["close"])

# rename Date to lowercase for consistency
prices_long = prices_long.rename(columns={"Date": "date"})

print(f"Rows after cleaning: {len(prices_long)}")
print(f"Unique tickers: {prices_long['ticker'].nunique()}")
print(f"Date range: {prices_long['date'].min()} to {prices_long['date'].max()}")
print(prices_long.head())

Rows after cleaning: 7286
Unique tickers: 102
Date range: 2020-01-01 00:00:00 to 2025-12-01 00:00:00
        date ticker      close
0 2020-01-01   AAPL  74.539894
1 2020-02-01   AAPL  65.833809
2 2020-03-01   AAPL  61.386501
3 2020-04-01   AAPL  70.924339
4 2020-05-01   AAPL  76.751854


In [118]:
all_tickers = universe["ticker"].tolist()
print(f"Pulling prices for {len(all_tickers)} tickers...")

prices_full = yf.download(
    all_tickers,
    start="2020-01-01",
    end="2025-12-31",
    interval="1mo"
)

print(f"Done. Raw shape: {prices_full.shape}")

Pulling prices for 102 tickers...


[*********************100%***********************]  102 of 102 completed

Done. Raw shape: (72, 510)


In [112]:
prices_long = (
    prices_test["Close"]
    .reset_index()
    .melt(id_vars="Date", var_name="ticker", value_name="close")
)

print("Long format:")
print(prices_long.head())
print(prices_long.shape)

Long format:
        Date ticker      close
0 2020-01-01   AAPL  74.539894
1 2020-02-01   AAPL  65.833809
2 2020-03-01   AAPL  61.386497
3 2020-04-01   AAPL  70.924347
4 2020-05-01   AAPL  76.751823
(360, 3)


In [113]:
close_wide = prices_test["Close"]
print("Wide format:")
print(close_wide.head())
print(close_wide.shape)

Wide format:
Ticker           AAPL         JNJ         JPM        MSFT        XOM
Date                                                                
2020-01-01  74.539894  125.074715  111.102402  161.262115  46.390259
2020-02-01  65.833809  112.984764   98.087936  153.475159  38.414604
2020-03-01  61.386497  110.872787   76.055969  149.809723  28.762501
2020-04-01  70.924347  126.861496   80.896561  170.232758  35.201302
2020-05-01  76.751823  125.770767   83.060219  174.070374  34.443798
(72, 5)


In [114]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [115]:
test_tickers = ["AAPL", "MSFT", "JPM", "XOM", "JNJ"]

prices_test = yf.download(
    test_tickers,
    start="2020-01-01",
    end="2025-12-31",
    interval="1mo"
)

print(prices_test.shape)
print(prices_test.head())

[*********************100%***********************]  5 of 5 completed

(72, 25)
Price           Close                                                 \
Ticker           AAPL         JNJ         JPM        MSFT        XOM   
Date                                                                   
2020-01-01  74.539879  125.074692  111.102394  161.262161  46.390263   
2020-02-01  65.833809  112.984795   98.087936  153.475143  38.414608   
2020-03-01  61.386509  110.872765   76.055946  149.809814  28.762497   
2020-04-01  70.924347  126.861519   80.896561  170.232758  35.201294   
2020-05-01  76.751839  125.770767   83.060196  174.070404  34.443790   

Price            High                                                 ...  \
Ticker           AAPL         JNJ         JPM        MSFT        XOM  ...   
Date                                                                  ...   
2020-01-01  78.956735  127.023870  118.438716  164.880928  53.298024  ...   
2020-02-01  78.805020  129.804814  117.670036  180.653727  47.054908  ...   
2020-03-01  73.386681  121.45

In [116]:
universe = pd.read_csv("/Users/jtee/Documents/us-equity-screener/data/universe.csv")
print(universe.head())
print(f"total rows: {len(universe)}")


  ticker          company_name                  sector
0   NVDA           NVIDIA CORP  Information Technology
1   AAPL             APPLE INC  Information Technology
2   MSFT        MICROSOFT CORP  Information Technology
3   AMZN        AMAZON.COM INC  Consumer Discretionary
4  GOOGL  ALPHABET INC CLASS A           Communication
total rows: 102


In [117]:
import yfinance as yf
import pandas as pd
import time